# 09 · Paket Data untuk Kaggle (Neural Network di GPU)

Neural network tabular terlalu lambat di CPU. Di mesin uji saya (2 core), satu fold RealMLP butuh lebih dari 40 menit, sehingga 15 fold bisa makan ±10 jam. Model-model ini dilatih di **notebook Kaggle dengan GPU**. Notebook ini (dijalankan **lokal**) menyiapkan paket yang perlu diunggah ke Kaggle sebagai *Dataset* privat.

| File di paket | Isi | Kenapa perlu |
|---|---|---|
| `train_v1.parquet`, `test_v1.parquet` | tabel fitur v1 (dari notebook 03, e-commerce **penuh**) | input model; urutan baris = urutan OOF |
| `features_v1.json` | daftar fitur & fitur kategorikal | definisi kolom yang sama |
| `folds.parquet` | nomor fold per baris untuk seed 42, 7, 2024, 11, 99 | fold **identik** dengan LightGBM, tanpa bergantung versi scikit-learn di Kaggle |
| `sample_submission.parquet` | ID test | validasi format |
| `modeling.py`, `nn_models.py` | kode bersama | pelatihan, metrik KS, format penyimpanan run yang sama |
| `lgbm_tuned_*` | OOF & prediksi test LightGBM tuned (notebook 06) | pratinjau korelasi & gabungan langsung di Kaggle |

Hasil dari Kaggle (OOF + prediksi test per model) diunduh kembali ke `data/preds/`, lalu digabung secara lokal di notebook 11.

In [1]:
import json
import shutil
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
import eda_utils as E
import modeling as M

BUNDLE = M.ROOT / "kaggle_bundle"
BUNDLE.mkdir(exist_ok=True)
meta = json.loads((M.FEAT_DIR / "features_v1.json").read_text())
train = pl.read_parquet(M.FEAT_DIR / "train_v1.parquet")
test = pl.read_parquet(M.FEAT_DIR / "test_v1.parquet")
print("e-commerce:", meta["ecommerce_source"], "| sampling:", meta["ecommerce_is_sampled"])
if meta["ecommerce_is_sampled"]:
    print("⚠️  Tabel fitur dibangun dari file e-commerce SAMPLING. Jalankan notebook 03 dengan file penuh dulu.")

e-commerce: ecommerce_transactions.csv | sampling: False


In [2]:
SEEDS = (42, 7, 2024, 11, 99)
folds = M.fold_table(train, seeds=SEEDS)
# verifikasi: fold sama persis dengan pembagian yang dipakai train_lgb_cv / train_cv
y = train["target"].to_numpy()
for s in SEEDS:
    for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=s).split(np.zeros(len(y)), y)):
        assert set(np.flatnonzero(folds[f"fold_s{s}"].to_numpy() == k + 1)) == set(va)
print("Tabel fold identik dengan pembagian LightGBM untuk semua seed ✔")
# verifikasi: OOF tersimpan lgbm_tuned memakai urutan baris yang sama
oof = pl.read_parquet(M.PRED_DIR / "lgbm_tuned_oof.parquet")
assert oof["application_id"].equals(train["application_id"])
print("Urutan baris = urutan OOF lgbm_tuned ✔")
folds.group_by("fold_s42").agg(n=pl.len()).sort("fold_s42")

Tabel fold identik dengan pembagian LightGBM untuk semua seed ✔
Urutan baris = urutan OOF lgbm_tuned ✔


fold_s42,n
i8,u32
1,14000
2,14000
3,14000
4,14000
5,14000


In [3]:
train.write_parquet(BUNDLE / "train_v1.parquet")
test.write_parquet(BUNDLE / "test_v1.parquet")
folds.write_parquet(BUNDLE / "folds.parquet")
shutil.copy(M.FEAT_DIR / "features_v1.json", BUNDLE / "features_v1.json")
shutil.copy(E.PQ_DIR / "sample_submission.parquet", BUNDLE / "sample_submission.parquet")
for f in ("modeling.py", "nn_models.py"):
    shutil.copy(Path.cwd() / f, BUNDLE / f)
for f in ("lgbm_tuned_oof.parquet", "lgbm_tuned_test.parquet", "lgbm_tuned_meta.json"):   # untuk pratinjau di Kaggle
    shutil.copy(M.PRED_DIR / f, BUNDLE / f)
for p in sorted(BUNDLE.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size / 1e6:7.1f} MB")

features_v1.json                 0.0 MB
folds.parquet                    0.8 MB
lgbm_tuned_meta.json             0.0 MB
lgbm_tuned_oof.parquet           3.8 MB
lgbm_tuned_test.parquet          0.5 MB
modeling.py                      0.0 MB
nn_models.py                     0.0 MB
sample_submission.parquet        0.3 MB
test_v1.parquet                 14.9 MB
train_v1.parquet                34.2 MB


## Langkah di Kaggle

1. **Unggah paket sebagai Dataset privat.** Buka kaggle.com → *Create* → *New Dataset*, seret **semua file** di folder `kaggle_bundle/`, beri nama misalnya `credit-risk-nn-bundle`, visibilitas **Private**, lalu *Create*.
2. **Buat notebook baru**, lalu impor `10_nn_kaggle.ipynb` (*File* → *Import Notebook*).
3. Di panel kanan notebook:
   - *Add Input* → pilih dataset `credit-risk-nn-bundle`.
   - *Accelerator* → **GPU T4 x2** atau **GPU P100**.
   - *Internet* → **On**, untuk `pip install` dan mengunduh checkpoint TabICL.
4. **Jalankan sebagai *commit*:** klik *Save Version* → *Save & Run All (Commit)*. Cara ini berjalan di latar belakang sampai 12 jam, tanpa terputus walau browser ditutup. Mode interaktif bisa terputus kalau tab tidak aktif.
   - Sebelumnya, untuk memastikan semua berjalan, Anda bisa menjalankan sel 1–3 secara interaktif dulu: instal, cek GPU, dan muat data.
   - Setiap model langsung disimpan setelah selesai.
5. Setelah commit selesai, buka versi notebook itu → tab *Output* → unduh `nn_runs.zip`, lalu ekstrak isinya ke folder `data/preds/` lokal Anda.
6. Jalankan notebook 11 secara lokal.

**Perkiraan waktu di GPU T4:** RealMLP ±1 jam, TabM ±0,5–1 jam (15 fold masing-masing), TabICL belum diketahui (5 fold). Waktu per fold dicetak, jadi sisa waktu bisa diperkirakan dari fold pertama.